In [ ]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join( "..")))
from turbulence_sr.dataloader.dataloader_3d import dataset_sr
from torch.utils.data import DataLoader, random_split
import torch

In [ ]:
dataset = dataset_sr(snapshot_index=79, use_normalizing=False)
dataloader = DataLoader(dataset)

In [ ]:
len(dataset)

In [ ]:
import matplotlib.pyplot as plt
n_plots = 3
fig, ax = plt.subplots(2, n_plots)

for i in range(n_plots):    
    ax[0][i].imshow(dataset[i][0][0,:,:,0])
    ax[1][i].imshow(dataset[i][1][0, :, :, 0])

In [ ]:
dataloader.mean_hr

In [ ]:
torch.mean(dataset[0][1])

In [ ]:
def compute_mean_std_dataloader(loader):
    n_channels = None
    n_samples = 0
    mean = 0.0
    var = 0.0

    for batch in loader:
        # Assuming your dataset returns (x, y) = (lr, hr) pairs
        x = batch[0]  # (B, C, D, H, W)
        if n_channels is None:
            n_channels = x.size(1)
            mean = torch.zeros(n_channels, device=x.device)
            var = torch.zeros(n_channels, device=x.device)

        # flatten spatial dimensions
        B, C, D, H, W = x.shape
        x = x.permute(1, 0, 2, 3, 4).reshape(C, -1)

        mean += x.mean(dim=1)
        var += x.var(dim=1, unbiased=False)
        n_samples += 1

    mean /= n_samples
    var /= n_samples
    std = torch.sqrt(var)

    return mean.cpu(), std.cpu()


In [ ]:
def compute_mean_std_dataset(dataset):
    n = len(dataset)
    first_x, _, _, _= dataset[0]  # (C, D, H, W)
    C = first_x.shape[0]

    mean = torch.zeros(C, dtype=torch.float64)
    var = torch.zeros(C, dtype=torch.float64)
    n_total = 0

    for i in range(n):
        x, _, _, _ = dataset[i]  # just LR or both LR/HR
        # flatten spatial
        x = x.view(C, -1)
        n_pixels = x.shape[1]

        mean += x.mean(dim=1) * n_pixels
        var += x.var(dim=1, unbiased=False) * n_pixels
        n_total += n_pixels

    mean /= n_total
    var /= n_total
    std = torch.sqrt(var)
    return mean.float(), std.float()


In [ ]:
import time

In [ ]:
start = time.time()
train_mean, train_std = compute_mean_std_dataset(dataset)
print(time.time() - start)
print("Train mean:", train_mean)
print("Train std:", train_std)


In [ ]:
start = time.time()
train_mean, train_std = compute_mean_std_dataloader(dataloader)
print(time.time() - start)
print("Train mean:", train_mean)
print("Train std:", train_std)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

num_channels = 5

# initialize lists to collect values per channel
values_highres = [[] for _ in range(num_channels)]
values_lowres = [[] for _ in range(num_channels)]

i = 0
# loop through dataset and collect voxel values
for data in dataset:
    highres, lowres = data[0], data[1]
    
    for c in range(num_channels):
        values_highres[c] = np.concatenate((values_highres[c], highres[c].ravel()))
        values_lowres[c] = np.concatenate((values_lowres[c], lowres[c].ravel()))

        i+= 1
    if i % 20 == 0:
        print(i, end = "\r")

# plot histograms
fig, axes = plt.subplots(2, num_channels, figsize=(4*num_channels, 8))

for c in range(num_channels):
    axes[0, c].hist(values_highres[c].numpy(), bins=100, color='blue', alpha=0.7)
    axes[0, c].set_title(f'High-res Channel {c}')
    axes[1, c].hist(values_lowres[c].numpy(), bins=100, color='blue', alpha=0.7)
    axes[1, c].set_title(f'Low-res Channel {c}')

    print(c, max(values_highres[c].numpy()))
plt.tight_layout()
plt.show()
